# Specimen_Findings — graph-fed consumer

Builds `sdtm-findings-graph/machine_actionable/Specimen_Findings.xlsx` for the
specimen sub-type of Findings. Two sheets:

| Sheet | Grain | Rows (expected) | Source |
|---|---|---|---|
| `Test_Identity` | TESTCD | ~4,183 | `SDTM_Test_Identity.xlsx` widened to specimen scope; COSMoS coverage flags from `DSS_View.xlsx` |
| `Measurement_Specs` | DSS | 160 | `DSS_View.xlsx` filtered to LB, MB, MI |

## Scope

Specimen-based Findings where specimen is the structural decomposition axis.
Behavioural rationale in `cosmos-bc-dss/docs/COSMoS_Behavioural_Analysis.md`.

**In scope:** LB, MB, MI, CP, BS, MS, PC, PP.
**Excluded:** IS (target-driven), GF (scale-driven), UR (no decomposition).

CP, BS, MS, PC, PP are in scope but have no published DSSs — they appear only
in `Test_Identity`.

## Inputs

| File | Track | Sheets used |
|---|---|---|
| `consumer-bases/interim/DSS_View.xlsx` | consumer-bases | `Test_Identity`, `Measurement_Specs` |
| `sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx` | sdtm-test-codes | `Test Codes` |
| `sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx` | sdtm-domain-reference | `Domains` |
| `cosmos-graph/interim/COSMoS_Graph_CT.xlsx` | cosmos-graph | `CodelistTerms` (for `Allowed_Units` expansion) |

## Output

`sdtm-findings-graph/machine_actionable/Specimen_Findings.xlsx`

## 1. Setup

In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

print('SPECIMEN_FINDINGS — graph-fed consumer')
print(f'Run: {datetime.now():%Y-%m-%d %H:%M}')

SPECIMEN_FINDINGS — graph-fed consumer
Run: 2026-04-26 12:00


In [2]:
BASE_DIR = Path.cwd().parent          # sdtm-findings-graph/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/

DSS_VIEW_FILE = REPO_ROOT / 'consumer-bases' / 'interim' / 'DSS_View.xlsx'
TEST_IDENTITY_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Test_Identity.xlsx'
DOMAIN_META_FILE = REPO_ROOT / 'sdtm-domain-reference' / 'machine_actionable' / 'SDTM_Domain_Metadata.xlsx'
GRAPH_CT_FILE = REPO_ROOT / 'cosmos-graph' / 'interim' / 'COSMoS_Graph_CT.xlsx'

OUTPUT_DIR = BASE_DIR / 'machine_actionable'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / 'Specimen_Findings.xlsx'

# Specimen-scope domains. Behavioural exclusions:
#   IS — target-driven, not specimen-driven (specimen constant; target mnemonic-encoded in DS_Code).
#   GF — scale-driven, not specimen-driven (specimen NCIt-encoded by legacy convention).
#   UR — zero decomposition despite Specimen_Based metadata flag.
SCOPE_DOMAINS = ['LB', 'MB', 'MI', 'CP', 'BS', 'MS', 'PC', 'PP']

# Allowed_Units expansion: expand ORRESU_codelist into permissible-value list only
# when the codelist has at most this many terms. UNIT (~950 terms) is too broad for
# per-row expansion; narrower sub-codelists are useful. Same threshold as Measurement_Findings.
ALLOWED_UNITS_THRESHOLD = 50

for f, label in [
    (DSS_VIEW_FILE, 'DSS_View'),
    (TEST_IDENTITY_FILE, 'SDTM_Test_Identity'),
    (DOMAIN_META_FILE, 'SDTM_Domain_Metadata'),
    (GRAPH_CT_FILE, 'COSMoS_Graph_CT'),
]:
    if not f.exists():
        raise FileNotFoundError(f'{label} not found: {f}')
    print(f'  {label}: {f.relative_to(REPO_ROOT)}')

print(f'  Output: {OUTPUT_FILE.relative_to(REPO_ROOT)}')
print(f'  Scope:  {SCOPE_DOMAINS}')
print(f'  Allowed_Units threshold: {ALLOWED_UNITS_THRESHOLD} terms')

  DSS_View: consumer-bases/interim/DSS_View.xlsx
  SDTM_Test_Identity: sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx
  SDTM_Domain_Metadata: sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx
  COSMoS_Graph_CT: cosmos-graph/interim/COSMoS_Graph_CT.xlsx
  Output: sdtm-findings-graph/machine_actionable/Specimen_Findings.xlsx
  Scope:  ['LB', 'MB', 'MI', 'CP', 'BS', 'MS', 'PC', 'PP']
  Allowed_Units threshold: 50 terms


## 2. Load inputs

In [3]:
view_ti = pd.read_excel(DSS_VIEW_FILE, sheet_name='Test_Identity', dtype=str).fillna('')
view_ms = pd.read_excel(DSS_VIEW_FILE, sheet_name='Measurement_Specs', dtype=str).fillna('')
ref_tests = pd.read_excel(TEST_IDENTITY_FILE, sheet_name='Test Codes', dtype=str).fillna('')
domain_meta = pd.read_excel(DOMAIN_META_FILE, sheet_name='Domains', dtype=str).fillna('')
ct_terms = pd.read_excel(GRAPH_CT_FILE, sheet_name='CodelistTerms', dtype=str).fillna('')

print(f'DSS_View Test_Identity:     {len(view_ti):>6,} rows  (COSMoS-pinned TESTCDs, graph-wide)')
print(f'DSS_View Measurement_Specs: {len(view_ms):>6,} rows  (DSS-grain, all 32 domains)')
print(f'SDTM_Test_Identity:         {len(ref_tests):>6,} rows  (TESTCDs scoped via SDTM_Domains)')
print(f'SDTM_Domain_Metadata:       {len(domain_meta):>6,} rows')
print(f'COSMoS CodelistTerms:       {len(ct_terms):>6,} rows')

DSS_View Test_Identity:        729 rows  (COSMoS-pinned TESTCDs, graph-wide)
DSS_View Measurement_Specs:  1,326 rows  (DSS-grain, all 32 domains)
SDTM_Test_Identity:          5,885 rows  (TESTCDs scoped via SDTM_Domains)
SDTM_Domain_Metadata:           57 rows
COSMoS CodelistTerms:       17,523 rows


## 3. Build Test_Identity

Universe = `SDTM_Test_Identity` rows where `SDTM_Domains` intersects the
specimen-scope domain set. The COSMoS-pinned subset is a flag (`Has_DSS`),
not a filter — preserves the coverage-gap framing.

COSMoS coverage aggregations (`DSS_Count`, `BC_Count`, `DS_Codes`, `BC_IDs`)
are computed over scope-domain DSSs only, not graph-wide.

In [4]:
# Filter SDTM_Test_Identity to TESTCDs that participate in any specimen-scope domain.
def in_scope(domains_str):
    if not domains_str:
        return False
    doms = {d.strip() for d in domains_str.split(';')}
    return bool(doms & set(SCOPE_DOMAINS))

ref_in_scope = ref_tests[ref_tests['SDTM_Domains'].apply(in_scope)].copy()
print(f"Specimen-scoped TESTCDs: {len(ref_in_scope):,}")

# In_Scope_Domains: filter SDTM_Domains down to only the scope members for each row.
def filter_to_scope(domains_str):
    doms = sorted({d.strip() for d in domains_str.split(';') if d.strip() in SCOPE_DOMAINS})
    return '; '.join(doms)

ref_in_scope['In_Scope_Domains'] = ref_in_scope['SDTM_Domains'].apply(filter_to_scope)

# Per-domain breakdown
print('Per-domain TESTCD count:')
for d in SCOPE_DOMAINS:
    n = ref_in_scope['SDTM_Domains'].apply(lambda s: d in [x.strip() for x in s.split(';')]).sum()
    print(f'  {d}: {n:>5,}')

Specimen-scoped TESTCDs: 4,183
Per-domain TESTCD count:
  LB: 2,505
  MB:   561
  MI:   254
  CP:   938
  BS:    30
  MS:    18
  PC:     0
  PP:     0


In [5]:
# COSMoS coverage from DSS_View Measurement_Specs, filtered to scope domains.
# This computes per-TESTCD aggregations over scope DSSs only — graph-wide aggregations
# from view_ti would over-count when a TESTCD also pins to non-scope domains.
view_ms_scope = view_ms[view_ms['domain'].isin(SCOPE_DOMAINS)].copy()
print(f"Specimen-scope DSS rows in DSS_View: {len(view_ms_scope):,}")

def _join_unique(s):
    return '; '.join(sorted({v for v in s if v}))

cosmos_cov = view_ms_scope.groupby('TESTCD_value', sort=True).agg(
    DSS_Count=('ds_id', 'nunique'),
    BC_Count=('bc_id', 'nunique'),
    DS_Codes=('ds_id', _join_unique),
    BC_IDs=('bc_id', _join_unique),
).reset_index().rename(columns={'TESTCD_value': 'TESTCD'})

cosmos_cov['DSS_Count'] = cosmos_cov['DSS_Count'].astype(str)
cosmos_cov['BC_Count'] = cosmos_cov['BC_Count'].astype(str)
print(f"COSMoS-pinned TESTCDs in scope: {len(cosmos_cov):,}")

# Conflict flags from DSS_View Test_Identity (TESTCD grain — graph-wide flags carry over).
flags = view_ti[['TESTCD', 'NCIt_Code_Conflict', 'NCIt_Reference_Disagree']].copy()

Specimen-scope DSS rows in DSS_View: 160
COSMoS-pinned TESTCDs in scope: 105


In [6]:
ti = ref_in_scope.copy()

# Has_DSS: derived membership in cosmos_cov (i.e. COSMoS pins this TESTCD in any scope domain).
ti['Has_DSS'] = ti['TESTCD'].isin(cosmos_cov['TESTCD']).map({True: 'Yes', False: 'No'})

# Join COSMoS coverage. Default counts to '0' for non-pinned TESTCDs.
ti = ti.merge(cosmos_cov, on='TESTCD', how='left').fillna('')
ti.loc[ti['DSS_Count'] == '', 'DSS_Count'] = '0'
ti.loc[ti['BC_Count'] == '', 'BC_Count'] = '0'

# Join NCIt conflict flags. Empty for non-pinned TESTCDs (only pinned TESTCDs can disagree).
ti = ti.merge(flags, on='TESTCD', how='left').fillna('')

# Final column order — keys, reference identity (green), COSMoS coverage (yellow), flags (grey).
TI_COLS = [
    'TESTCD', 'NCIt_Code', 'In_Scope_Domains', 'SDTM_Domains',
    'TEST', 'NCIt_Preferred_Term', 'NCIt_Synonyms', 'NCIt_Definition',
    'UMLS_CUI', 'NCIm_CUI',
    'Has_DSS', 'DSS_Count', 'BC_Count', 'DS_Codes', 'BC_IDs',
    'NCIt_Code_Conflict', 'NCIt_Reference_Disagree',
]

missing = [c for c in TI_COLS if c not in ti.columns]
if missing:
    raise RuntimeError(f'Missing expected Test_Identity columns: {missing}')

ti_final = ti[TI_COLS].copy()
print(f"Test_Identity: {len(ti_final):,} rows x {len(ti_final.columns)} cols")
print(f"  With COSMoS coverage (Has_DSS=Yes): {(ti_final['Has_DSS'] == 'Yes').sum():,}")
print(f"  Coverage gap   (Has_DSS=No):        {(ti_final['Has_DSS'] == 'No').sum():,}")
print(f"  NCIt_Code_Conflict=Yes:             {(ti_final['NCIt_Code_Conflict'] == 'Yes').sum()}")
print(f"  NCIt_Reference_Disagree=Yes:        {(ti_final['NCIt_Reference_Disagree'] == 'Yes').sum()}")

Test_Identity: 4,183 rows x 17 cols
  With COSMoS coverage (Has_DSS=Yes): 104
  Coverage gap   (Has_DSS=No):        4,079
  NCIt_Code_Conflict=Yes:             0
  NCIt_Reference_Disagree=Yes:        0


## 4. Build Measurement_Specs

Filter `DSS_View.Measurement_Specs` to specimen-scope domains. Add:

- `Observation_Class` join from `SDTM_Domain_Metadata`.
- `LOINC_DSS` (= `LOINC_value`, DSS-grain pin — primary).
- `LOINC_BC` (coalesce of `Coding_http://loinc.org/` and `Coding_https://loinc.org`
  — BC-grain, with the source-data URI inconsistency flattened).

Validation checks (TESTCD_NCIt ↔ BC NCIt consistency, Quantitative-without-units,
etc.) belong in a separate validation step, not in this consumer projection.
See `cosmos-bc-dss/notebooks/COSMoS_BC_DSS_Validate.ipynb` for the legacy QC pattern.

Rename `TESTCD_value` → `TESTCD`, `TEST_value` → `TEST`, `TESTCD_ncit` → `TESTCD_NCIt`
for join symmetry with the `Test_Identity` sheet. Other pin columns keep their
`<remainder>_value/_ncit/_codelist` triplet naming.

In [7]:
ms = view_ms[view_ms['domain'].isin(SCOPE_DOMAINS)].copy()
print(f"Measurement_Specs rows in scope: {len(ms):,}")
print(f"  Domains present:           {sorted(ms['domain'].unique())}")
print(f"  Domains with no DSSs yet:  {sorted(set(SCOPE_DOMAINS) - set(ms['domain'].unique()))}")

Measurement_Specs rows in scope: 160
  Domains present:           ['LB', 'MB', 'MI']
  Domains with no DSSs yet:  ['BS', 'CP', 'MS', 'PC', 'PP']


In [8]:
# Observation_Class join.
ms = ms.merge(
    domain_meta[['Domain', 'Observation_Class']].rename(columns={'Domain': 'domain'}),
    on='domain', how='left',
  ).fillna('')

# LOINC_BC: coalesce the two source URI variants. The graph projects the URI
# inconsistency in the BC Coding list verbatim (see cosmos-graph/docs/COSMoS_Graph.md).
LOINC_BC_SOURCES = ['Coding_http://loinc.org/', 'Coding_https://loinc.org']
def _coalesce_loinc(row):
    parts = set()
    for col in LOINC_BC_SOURCES:
        v = row.get(col, '')
        if v:
            for p in v.split(';'):
                p = p.strip()
                if p:
                    parts.add(p)
    return '; '.join(sorted(parts))

ms['LOINC_BC'] = ms.apply(_coalesce_loinc, axis=1)
print(f'LOINC_BC populated rows: {(ms["LOINC_BC"] != "").sum()}')

LOINC_BC populated rows: 11


In [9]:
# Allowed_Units: conditional expansion of ORRESU_codelist when the codelist is narrow
# enough that per-row expansion is signal, not noise. For specimen, ORRESU_codelist is
# uniformly UNIT (~950 terms), so this fires zero rows — the column will be empty.
# Kept for shape parity with Measurement_Findings, where VSRESU (~29 terms) does fire.
codelist_size = ct_terms.groupby('codelist_submission_value')['term_submission_value'].nunique()
narrow_codelists = set(codelist_size[codelist_size <= ALLOWED_UNITS_THRESHOLD].index)
print(f'Codelists with <= {ALLOWED_UNITS_THRESHOLD} terms: {len(narrow_codelists):,}')

terms_by_codelist = {}
for cl, group in ct_terms.groupby('codelist_submission_value'):
    if cl in narrow_codelists:
        terms = sorted({t for t in group['term_submission_value'] if t})
        terms_by_codelist[cl] = '; '.join(terms)

ms['Allowed_Units'] = ms['ORRESU_codelist'].map(terms_by_codelist).fillna('')
expanded = (ms['Allowed_Units'] != '').sum()
print(f'Rows with Allowed_Units expansion: {expanded:,}')
if expanded:
    fired_codelists = sorted({cl for cl in ms['ORRESU_codelist'] if cl in narrow_codelists})
    print(f'  Fired codelists: {fired_codelists}')

Codelists with <= 50 terms: 244
Rows with Allowed_Units expansion: 0


In [10]:
# Rename DSS_View native columns to consumer link-key naming.
# Local exception to the leverage-DSS_View-naming rule, for join symmetry with Test_Identity.
ms = ms.rename(columns={
    'TESTCD_value': 'TESTCD',
    'TEST_value':   'TEST',
    'TESTCD_ncit':  'TESTCD_NCIt',
    'LOINC_value':  'LOINC_DSS',
})

In [11]:
# PIN_SCHEMA declares which pin remainders this consumer emits, classified by policy:
#   core     — column always emitted, even when no row fires (empty column).
#              Reserved for first-order structural pins (decomposition axes).
#   optional — column emitted only when at least one row fires.
#              Includes qualifier pins (units, location, evaluator, test detail) —
#              their presence is a consequence of other columns (e.g. ORRESU/STRESU
#              presence is conditional on result_scales including Quantitative).
#   exclude  — pin handled elsewhere (link-key, LOINC, or N/A for this sub-type).
# Drift detection: pins firing in source but not declared get a WARN at runtime.
PIN_SCHEMA = {
    'core':     ['SPEC', 'METHOD', 'CAT'],
    'optional': ['ORRESU', 'STRESU', 'LOC', 'TSTDTL', 'EVAL'],
    'exclude':  ['TESTCD', 'TEST', 'LOINC'],
}

def fires(remainder):
    for suffix in ('_value', '_ncit', '_codelist'):
        col = f'{remainder}{suffix}'
        if col in ms.columns and (ms[col] != '').any():
            return True
    return False

declared = set(PIN_SCHEMA['core']) | set(PIN_SCHEMA['optional']) | set(PIN_SCHEMA['exclude'])
all_candidate_remainders = sorted({c.rsplit('_', 1)[0] for c in ms.columns if c.endswith('_value')})

pins_to_emit = []
notes = []

# Core: always emit, in declared order. Warn if column not present in source.
for r in PIN_SCHEMA['core']:
    if f'{r}_value' not in ms.columns:
        notes.append(f"WARN: core pin '{r}' has no columns in DSS_View — schema likely needs updating.")
        continue
    pins_to_emit.append(r)
    if not fires(r):
        notes.append(f"INFO: core pin '{r}' not firing on this scope — schema preserved, column will be empty.")

# Optional: emit only when firing, in declared order.
for r in PIN_SCHEMA['optional']:
    if fires(r):
        pins_to_emit.append(r)

# Undeclared firing pins: WARN and emit at end (drift detection).
for r in all_candidate_remainders:
    if r in declared:
        continue
    if fires(r):
        n_rows = (ms[f'{r}_value'] != '').sum()
        notes.append(f"WARN: undeclared pin '{r}' firing on {n_rows} rows — add to PIN_SCHEMA core/optional/exclude.")
        pins_to_emit.append(r)

for note in notes:
    print(note)
if notes:
    print()

print(f"Pins to emit ({len(pins_to_emit)}, in order): {pins_to_emit}")
print(f"  Core (always present):  {[r for r in pins_to_emit if r in PIN_SCHEMA['core']]}")
print(f"  Optional (firing-only): {[r for r in pins_to_emit if r in PIN_SCHEMA['optional']]}")
print(f"  Undeclared (firing):    {[r for r in pins_to_emit if r not in declared]}")

Pins to emit (7, in order): ['SPEC', 'METHOD', 'CAT', 'ORRESU', 'STRESU', 'LOC', 'TSTDTL']
  Core (always present):  ['SPEC', 'METHOD', 'CAT']
  Optional (firing-only): ['ORRESU', 'STRESU', 'LOC', 'TSTDTL']
  Undeclared (firing):    []


In [12]:
# Assemble final column order — keys, test identity, BC identity, DSS identity,
# firing-pin triplets, LOINC pair, Allowed_Units, conflict flag.
identity_cols = [
    'domain', 'Observation_Class', 'ds_id', 'bc_id', 'TESTCD',
    'TEST', 'TESTCD_NCIt',
    'bc_short_name', 'bc_definition', 'bc_categories', 'bc_parent_label',
    'bc_hierarchy_path', 'bc_type', 'result_scales', 'bc_ncit_code',
    'ds_short_name', 'sdtmig_start_version', 'sdtmig_end_version',
    'package_date', 'source',
]

pin_cols = []
for r in pins_to_emit:
    for suffix in ('_value', '_ncit', '_codelist'):
        col = f'{r}{suffix}'
        if col in ms.columns:
            pin_cols.append(col)

loinc_cols = ['LOINC_DSS', 'LOINC_BC']
tail_cols = ['Allowed_Units']

MS_COLS = identity_cols + pin_cols + loinc_cols + tail_cols

missing = [c for c in MS_COLS if c not in ms.columns]
if missing:
    raise RuntimeError(f'Missing expected Measurement_Specs columns: {missing}')

ms_final = ms[MS_COLS].copy()
print(f"Measurement_Specs: {len(ms_final):,} rows x {len(ms_final.columns)} cols")
print(f"  Identity: {len(identity_cols)}  Pins: {len(pin_cols)}  LOINC: {len(loinc_cols)}  Tail: {len(tail_cols)}")

Measurement_Specs: 160 rows x 44 cols
  Identity: 20  Pins: 21  LOINC: 2  Tail: 1


## 5. Write workbook

Three sheets: `ReadMe`, `Test_Identity`, `Measurement_Specs`.

Header colour convention (per repo standard):

- **Green** — TESTCD / SDTM-CT-side columns (NCI EVS reference identity).
- **Yellow** — COSMoS-side columns (BC, DSS, pin pivot).
- **Grey** — keys, conflict flags, aggregation columns.

In [13]:
HEADER_FONT = Font(name='Arial', bold=True, size=10, color='FFFFFF')
DATA_FONT = Font(name='Arial', size=10)
WRAP = Alignment(wrap_text=True, vertical='top')

GREEN_HEADER  = PatternFill('solid', fgColor='548235')
YELLOW_HEADER = PatternFill('solid', fgColor='FFD700')
GREY_HEADER   = PatternFill('solid', fgColor='808080')


def write_sheet(ws, df, header_fills, col_widths):
    """Write a DataFrame; header_fills maps col_name -> PatternFill;
    col_widths maps col_name -> width (default 18 if missing)."""
    cols = list(df.columns)
    for ci, name in enumerate(cols, 1):
        cell = ws.cell(row=1, column=ci, value=name)
        cell.font = HEADER_FONT
        cell.fill = header_fills.get(name, GREY_HEADER)
        cell.alignment = WRAP
    for ri, (_, row) in enumerate(df.iterrows(), 2):
        for ci, name in enumerate(cols, 1):
            val = row[name]
            cell = ws.cell(row=ri, column=ci, value=val if val != '' else None)
            cell.font = DATA_FONT
            cell.alignment = WRAP
    for ci, name in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(ci)].width = col_widths.get(name, 18)
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = f'A1:{get_column_letter(len(cols))}1'

print('Style helpers ready.')

Style helpers ready.


In [14]:
wb = Workbook()

# ── ReadMe sheet ──
ws_rm = wb.active
ws_rm.title = 'ReadMe'

readme_font = Font(name='Arial', size=10)
title_font = Font(name='Arial', size=12, bold=True)
section_font = Font(name='Arial', size=10, bold=True)

readme_lines = [
    ('Specimen_Findings — graph-fed consumer', title_font),
    ('', None),
    ('PROVENANCE', section_font),
    (f'Generated: {datetime.now():%Y-%m-%d %H:%M}', readme_font),
    ('Notebook: sdtm-findings-graph/notebooks/Specimen_Findings.ipynb', readme_font),
    ('Track:    sdtm-findings-graph (graph-fed Findings consumer)', readme_font),
    ('Inputs:', readme_font),
    ('  consumer-bases/interim/DSS_View.xlsx (Test_Identity, Measurement_Specs)', readme_font),
    ('  sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx (Test Codes)', readme_font),
    ('  sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx (Domains)', readme_font),
    ('', None),
    ('SCOPE', section_font),
    ('Specimen-based Findings where specimen is the structural decomposition axis.', readme_font),
    ('In scope: LB, MB, MI, CP, BS, MS, PC, PP.', readme_font),
    ('Excluded: IS (target-driven), GF (scale-driven), UR (no decomposition).', readme_font),
    ('Behavioural rationale: cosmos-bc-dss/docs/COSMoS_Behavioural_Analysis.md', readme_font),
    ('', None),
    ('SHEETS', section_font),
    ('Test_Identity — one row per TESTCD scoped to specimen domains. Universe is', readme_font),
    ('  the wider SDTM_Test_Identity (NCI EVS), filtered to scope. Has_DSS flags', readme_font),
    ('  whether COSMoS pins the TESTCD. DSS_Count / BC_Count / DS_Codes / BC_IDs', readme_font),
    ('  are aggregated over scope DSSs only (not graph-wide).', readme_font),
    ('Measurement_Specs — one row per DSS in LB, MB, MI. CP, BS, MS, PC, PP are in', readme_font),
    ('  scope but have no DSSs yet — they appear only in Test_Identity.', readme_font),
    ('', None),
    ('CONSUMER-OWED DECISIONS APPLIED', section_font),
    ('LOINC: surfaced at both grains. LOINC_DSS = pinned LOINC at the DSS-Variable', readme_font),
    ('  level (richer for specimen). LOINC_BC = pinned LOINC at the BC level,', readme_font),
    ('  coalesced across the two source URI variants (graph projects this URI', readme_font),
    ('  inconsistency verbatim — see cosmos-graph/docs/COSMoS_Graph.md).', readme_font),
    ('Allowed_Units expansion: applied — column kept for shape parity with', readme_font),
    ('  Measurement_Findings. ORRESU_codelist is expanded against CodelistTerms only', readme_font),
    (f'  when the codelist has <= {ALLOWED_UNITS_THRESHOLD} terms. For specimen, ORRESU_codelist is', readme_font),
    ('  uniformly UNIT (~950 terms), so the column is empty across all rows.', readme_font),
    ('  ORRESU_value carries the pinned unit per DSS where one is set.', readme_font),
    ('TESTCD universe widening: yes — Test_Identity uses the full SDTM_Test_Identity', readme_font),
    ('  universe scoped to specimen domains, not just COSMoS-pinned TESTCDs.', readme_font),
    ('Domain class: Observation_Class joined per Measurement_Specs row.', readme_font),
    ('', None),
    ('CONFLICT FLAGS', section_font),
    ('NCIt_Code_Conflict (Test_Identity) — multiple NCIt concepts resolve from the', readme_font),
    ('  same TESTCD across the graph.', readme_font),
    ('NCIt_Reference_Disagree (Test_Identity) — NCIt at TESTCD grain disagrees', readme_font),
    ('  between graph (COSMoS pin) and reference (SDTM_Test_Identity / NCI EVS).', readme_font),
    ('Validation checks (e.g. TESTCD_NCIt vs BC NCIt_Code, Quantitative-without-units)', readme_font),
    ('  belong in a separate validation step, not in this consumer projection.', readme_font),
    ('  See cosmos-bc-dss/notebooks/COSMoS_BC_DSS_Validate.ipynb for the legacy QC', readme_font),
    ('  pattern; a graph-fed equivalent is planned.', readme_font),
    ('', None),
    ('HEADER COLOUR CONVENTION', section_font),
    ('Green  = TESTCD / SDTM-CT-side columns (NCI EVS reference identity).', readme_font),
    ('Yellow = COSMoS-side columns (BC, DSS, pin pivot).', readme_font),
    ('Grey   = keys, conflict flags, aggregation columns.', readme_font),
    ('', None),
    ('STATUS', section_font),
    ('Specimen sub-type — first build of the graph-fed Findings consumer.', readme_font),
    ('Parallel to the legacy sdtm-findings/Specimen_Findings.xlsx, which reads', readme_font),
    ('cosmos-bc-dss/interim/COSMoS_BC_DSS.xlsx. The legacy retires once all three', readme_font),
    ('sub-types (specimen, measurement, instrument) are built here.', readme_font),
    ('Not an official CDISC product.', readme_font),
]

for ri, (text, font) in enumerate(readme_lines, 1):
    cell = ws_rm.cell(row=ri, column=1, value=text if text else None)
    if font:
        cell.font = font

ws_rm.column_dimensions['A'].width = 100
print(f"ReadMe: {len(readme_lines)} lines")

ReadMe: 60 lines


In [15]:
# ── Test_Identity sheet ──
ws_ti = wb.create_sheet('Test_Identity')

TI_FILLS = {
    # Keys (grey)
    'TESTCD':                 GREY_HEADER,
    'NCIt_Code':              GREY_HEADER,
    'In_Scope_Domains':       GREY_HEADER,
    'SDTM_Domains':           GREY_HEADER,
    # Reference identity (green)
    'TEST':                   GREEN_HEADER,
    'NCIt_Preferred_Term':    GREEN_HEADER,
    'NCIt_Synonyms':          GREEN_HEADER,
    'NCIt_Definition':        GREEN_HEADER,
    'UMLS_CUI':               GREEN_HEADER,
    'NCIm_CUI':               GREEN_HEADER,
    # COSMoS coverage (yellow)
    'Has_DSS':                YELLOW_HEADER,
    'DSS_Count':              YELLOW_HEADER,
    'BC_Count':               YELLOW_HEADER,
    'DS_Codes':               YELLOW_HEADER,
    'BC_IDs':                 YELLOW_HEADER,
    # Conflict flags (grey)
    'NCIt_Code_Conflict':       GREY_HEADER,
    'NCIt_Reference_Disagree':  GREY_HEADER,
}

TI_WIDTHS = {
    'TESTCD': 14, 'NCIt_Code': 12, 'In_Scope_Domains': 18, 'SDTM_Domains': 22,
    'TEST': 38, 'NCIt_Preferred_Term': 38, 'NCIt_Synonyms': 50,
    'NCIt_Definition': 60, 'UMLS_CUI': 12, 'NCIm_CUI': 12,
    'Has_DSS': 9, 'DSS_Count': 9, 'BC_Count': 9, 'DS_Codes': 30, 'BC_IDs': 30,
    'NCIt_Code_Conflict': 12, 'NCIt_Reference_Disagree': 14,
}

write_sheet(ws_ti, ti_final, TI_FILLS, TI_WIDTHS)
print(f"Test_Identity: {len(ti_final):,} rows x {len(ti_final.columns)} cols")

Test_Identity: 4,183 rows x 17 cols


In [16]:
# ── Measurement_Specs sheet ──
ws_ms = wb.create_sheet('Measurement_Specs')

# Default everything to yellow (COSMoS-side); override grey for keys/flags, green for test identity.
MS_FILLS = {c: YELLOW_HEADER for c in ms_final.columns}
MS_FILLS.update({
    # Keys (grey)
    'domain': GREY_HEADER, 'Observation_Class': GREY_HEADER,
    'ds_id': GREY_HEADER, 'bc_id': GREY_HEADER, 'TESTCD': GREY_HEADER,
    # Test identity (green)
    'TEST': GREEN_HEADER, 'TESTCD_NCIt': GREEN_HEADER,
})

MS_WIDTHS = {
    'domain': 8, 'Observation_Class': 14, 'ds_id': 16, 'bc_id': 14, 'TESTCD': 14,
    'TEST': 38, 'TESTCD_NCIt': 12,
    'bc_short_name': 32, 'bc_definition': 50, 'bc_categories': 35,
    'bc_parent_label': 30, 'bc_hierarchy_path': 45, 'bc_type': 12,
    'result_scales': 18, 'bc_ncit_code': 12,
    'ds_short_name': 38, 'sdtmig_start_version': 12, 'sdtmig_end_version': 12,
    'package_date': 12, 'source': 16,
    'SPEC_value': 14, 'SPEC_ncit': 12, 'SPEC_codelist': 12,
    'METHOD_value': 22, 'METHOD_ncit': 12, 'METHOD_codelist': 14,
    'ORRESU_value': 12, 'ORRESU_ncit': 12, 'ORRESU_codelist': 14,
    'STRESU_value': 12, 'STRESU_ncit': 12, 'STRESU_codelist': 14,
    'LOINC_DSS': 14, 'LOINC_BC': 26,
    'TSTDTL_value': 22, 'TSTDTL_ncit': 12, 'TSTDTL_codelist': 14,
    'LOC_value': 18, 'LOC_ncit': 12, 'LOC_codelist': 12,
    'CAT_value': 22, 'CAT_ncit': 12, 'CAT_codelist': 14,
    'Allowed_Units': 50,
}

write_sheet(ws_ms, ms_final, MS_FILLS, MS_WIDTHS)
print(f"Measurement_Specs: {len(ms_final):,} rows x {len(ms_final.columns)} cols")

Measurement_Specs: 160 rows x 44 cols


In [17]:
wb.save(OUTPUT_FILE)
print(f"\nWritten: {OUTPUT_FILE}")
print(f"File size: {OUTPUT_FILE.stat().st_size / 1024:.0f} KB")


Written: /sessions/affectionate-fervent-gauss/mnt/cdisc-for-ai/sdtm-findings-graph/machine_actionable/Specimen_Findings.xlsx
File size: 497 KB


## 6. Summary

In [18]:
print('=== Specimen_Findings summary ===')
print(f'Output: {OUTPUT_FILE.relative_to(REPO_ROOT)}')
print()
print(f'Test_Identity:    {len(ti_final):>5,} rows x {len(ti_final.columns):>2} cols')
print(f'  COSMoS-pinned (Has_DSS=Yes):  {(ti_final["Has_DSS"] == "Yes").sum():>5,}')
print(f'  Coverage gap (Has_DSS=No):    {(ti_final["Has_DSS"] == "No").sum():>5,}')
print(f'  NCIt_Code_Conflict=Yes:       {(ti_final["NCIt_Code_Conflict"] == "Yes").sum():>5}')
print(f'  NCIt_Reference_Disagree=Yes:  {(ti_final["NCIt_Reference_Disagree"] == "Yes").sum():>5}')
print()
print(f'Measurement_Specs: {len(ms_final):>4,} rows x {len(ms_final.columns):>2} cols')
print(f'  Domains:                     {sorted(ms_final["domain"].unique())}')
print(f'  Pins emitted:                {pins_to_emit}')
print(f'  Allowed_Units expanded:      {(ms_final["Allowed_Units"] != "").sum()} rows')

=== Specimen_Findings summary ===
Output: sdtm-findings-graph/machine_actionable/Specimen_Findings.xlsx

Test_Identity:    4,183 rows x 17 cols
  COSMoS-pinned (Has_DSS=Yes):    104
  Coverage gap (Has_DSS=No):    4,079
  NCIt_Code_Conflict=Yes:           0
  NCIt_Reference_Disagree=Yes:      0

Measurement_Specs:  160 rows x 44 cols
  Domains:                     ['LB', 'MB', 'MI']
  Pins emitted:                ['SPEC', 'METHOD', 'CAT', 'ORRESU', 'STRESU', 'LOC', 'TSTDTL']
  Allowed_Units expanded:      0 rows
